In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Running with catalog={catalog}, schema={schema}")

In [0]:


from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
import json


@dataclass
class Transaction:
    transaction_id: str
    step: int
    type: str                     # CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER
    amount: float
    name_orig: str
    oldbalance_org: float
    newbalance_orig: float
    name_dest: str
    oldbalance_dest: float
    newbalance_dest: float
    is_fraud: int
    is_flagged_fraud: int = 0
    event_timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError(f"amount cannot be negative: {self.amount}")
        if self.is_fraud not in (0, 1):
            raise ValueError(f"is_fraud must be 0 or 1, got {self.is_fraud}")

    def to_dict(self) -> dict:
        return asdict(self)

    def to_json(self) -> str:
        return json.dumps(self.to_dict())

    @classmethod
    def from_paysim_row(cls, row: dict, transaction_id: str) -> "Transaction":
        return cls(
            transaction_id=transaction_id,
            step=int(row["step"]),
            type=str(row["type"]),
            amount=float(row["amount"]),
            name_orig=str(row["nameOrig"]),
            oldbalance_org=float(row["oldbalanceOrg"]),
            newbalance_orig=float(row["newbalanceOrig"]),
            name_dest=str(row["nameDest"]),
            oldbalance_dest=float(row["oldbalanceDest"]),
            newbalance_dest=float(row["newbalanceDest"]),
            is_fraud=int(row["isFraud"]),
            is_flagged_fraud=int(row.get("isFlaggedFraud", 0)),
        )

In [0]:
t = Transaction(
    transaction_id="txn_000001",
    step=1,
    type="PAYMENT",
    amount=9839.64,
    name_orig="C1231006815",
    oldbalance_org=170136.0,
    newbalance_orig=160296.36,
    name_dest="M1979787155",
    oldbalance_dest=0.0,
    newbalance_dest=0.0,
    is_fraud=0,
)
print(t.to_json())

In [0]:
# Cell: Load PaySim dataset into Spark DataFrame

csv_path = "/Volumes/workspace/default/fraudguard_data/PS_20174392719_1491204439457_log.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

print(f"Total rows: {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(5)